In [2]:
import pandas as pd

pd.options.display.float_format = '{:,.0f}'.format

# Carregar os três arquivos de 2014
dre = pd.read_csv("../DRE_Tratado/dfp_dre_2014_filtrado.csv",            sep=";", encoding="utf-8")
bpa = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2014_filtrado.csv",   sep=";", encoding="utf-8")
bpp = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2014_filtrado.csv", sep=";", encoding="utf-8")

# Filtrar setor financeiro
dre_fin = dre[dre['setor'] == 'Financeiro']
bpa_fin = bpa[bpa['setor'] == 'Financeiro']
bpp_fin = bpp[bpp['setor'] == 'Financeiro']

# Confirmar Patrimônio Líquido
pl = bpp_fin[bpp_fin['codigo_conta'] == '2.03'][['empresa', 'codigo_conta', 'descricao_conta', 'valor']]
print(pl.to_string())


                          empresa codigo_conta                           descricao_conta         valor
3                 BCO BRASIL S.A.         2.03  Passivos Financeiros ao Custo Amortizado 1,105,625,511
1142          BCO ABC BRASIL S.A.         2.03  Passivos Financeiros ao Custo Amortizado    15,860,133
1186         BCO BTG PACTUAL S.A.         2.03  Passivos Financeiros ao Custo Amortizado    83,647,806
1334                 BCO PAN S.A.         2.03  Passivos Financeiros ao Custo Amortizado    18,109,743
1384            BCO BRADESCO S.A.         2.03  Passivos Financeiros ao Custo Amortizado   757,383,017
1431   ITAU UNIBANCO HOLDING S.A.         2.03  Passivos Financeiros ao Custo Amortizado   779,284,000
1591  BCO SANTANDER (BRASIL) S.A.         2.03  Passivos Financeiros ao Custo Amortizado   392,186,593


In [3]:
bb = bpp_fin[bpp_fin['empresa'] == 'BCO BRASIL S.A.'][['codigo_conta', 'descricao_conta', 'valor']]
print(bb.sort_values('codigo_conta').to_string())


   codigo_conta                                               descricao_conta         valor
0             2                                                 Passivo Total 1,278,136,948
1          2.01                          Passivos Financeiros para Negociação             0
2          2.02       Outros Passivos Financeiros ao Valor Justo no Resultado     2,995,367
3          2.03                      Passivos Financeiros ao Custo Amortizado 1,105,625,511
4       2.03.01                                         Depósitos de Clientes   437,821,753
5       2.03.02                    Valores a Pagar a Instituições Financeiras    30,675,249
6       2.03.03                       Obrigações por Operações Compromissadas   293,920,434
7       2.03.04                                     Obrigações de Curto Prazo    20,327,943
8       2.03.05                                     Obrigações de Longo Prazo   322,880,132
9          2.04                                                     Provisões   

In [4]:
pl = bpp_fin[bpp_fin['codigo_conta'] == '2.08'][['empresa', 'codigo_conta', 'descricao_conta', 'valor']]
print(pl.to_string())


                          empresa codigo_conta                 descricao_conta       valor
17                BCO BRASIL S.A.         2.08  Patrimônio Líquido Consolidado  85,440,036
1155          BCO ABC BRASIL S.A.         2.08  Patrimônio Líquido Consolidado   2,284,769
1197         BCO BTG PACTUAL S.A.         2.08  Patrimônio Líquido Consolidado  15,495,119
1353                 BCO PAN S.A.         2.08  Patrimônio Líquido Consolidado   3,643,526
1400            BCO BRADESCO S.A.         2.08  Patrimônio Líquido Consolidado  82,291,805
1449   ITAU UNIBANCO HOLDING S.A.         2.08  Patrimônio Líquido Consolidado 100,617,000
1611  BCO SANTANDER (BRASIL) S.A.         2.08  Patrimônio Líquido Consolidado  78,683,293


In [5]:
# Extrair Lucro Líquido da DRE
ll = dre_fin[dre_fin['codigo_conta'] == '3.09.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})

# Extrair Patrimônio Líquido do BPP
pl = bpp_fin[bpp_fin['codigo_conta'] == '2.08'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})

# Juntar os dois
kpis = ll.merge(pl, on='empresa')

# Calcular ROE
kpis['roe'] = kpis['lucro_liquido'] / kpis['patrimonio_liquido']

# Adicionar setor e tipo
mapa = {
    'BCO BRASIL S.A.':              ('Financeiro', 'Estatal'),
    'ITAU UNIBANCO HOLDING S.A.':   ('Financeiro', 'Privado'),
    'BCO BRADESCO S.A.':            ('Financeiro', 'Privado'),
    'BCO SANTANDER (BRASIL) S.A.':  ('Financeiro', 'Privado'),
    'BCO BTG PACTUAL S.A.':         ('Financeiro', 'Privado'),
    'BCO ABC BRASIL S.A.':          ('Financeiro', 'Privado'),
    'BCO PAN S.A.':                 ('Financeiro', 'Privado'),
}
kpis['setor'] = kpis['empresa'].map(lambda x: mapa[x][0])
kpis['tipo']  = kpis['empresa'].map(lambda x: mapa[x][1])
kpis['ano']   = 2014

# Exibir
pd.options.display.float_format = '{:.4f}'.format
print(kpis[['empresa', 'tipo', 'lucro_liquido', 'patrimonio_liquido', 'roe']].sort_values('roe', ascending=False).to_string())

                       empresa     tipo  lucro_liquido  patrimonio_liquido    roe
5   ITAU UNIBANCO HOLDING S.A.  Privado  21555000.0000      100617000.0000 0.2142
4            BCO BRADESCO S.A.  Privado  15314943.0000       82291805.0000 0.1861
1          BCO ABC BRASIL S.A.  Privado    328379.0000        2284769.0000 0.1437
0              BCO BRASIL S.A.  Estatal  11853096.0000       85440036.0000 0.1387
2         BCO BTG PACTUAL S.A.  Privado   2117086.0000       15495119.0000 0.1366
6  BCO SANTANDER (BRASIL) S.A.  Privado   5630023.0000       78683293.0000 0.0716
3                 BCO PAN S.A.  Privado     82516.0000        3643526.0000 0.0226


In [6]:
# Extrair Receita
receita = dre_fin[dre_fin['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})

# Juntar com kpis
kpis = kpis.merge(receita, on='empresa')

# Calcular Margem Líquida
kpis['margem_liquida'] = kpis['lucro_liquido'] / kpis['receita']

print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida']].sort_values('margem_liquida', ascending=False).to_string())

                       empresa     tipo    roe  margem_liquida
1          BCO ABC BRASIL S.A.  Privado 0.1437          0.1970
2         BCO BTG PACTUAL S.A.  Privado 0.1366          0.1821
5   ITAU UNIBANCO HOLDING S.A.  Privado 0.2142          0.1670
4            BCO BRADESCO S.A.  Privado 0.1861          0.1536
6  BCO SANTANDER (BRASIL) S.A.  Privado 0.0716          0.0955
0              BCO BRASIL S.A.  Estatal 0.1387          0.0860
3                 BCO PAN S.A.  Privado 0.0226          0.0120


In [7]:
# Extrair todas as contas necessárias
ll           = dre_fin[dre_fin['codigo_conta'] == '3.09.01'][['empresa', 'valor']].rename(columns={'valor': 'lucro_liquido'})
pl           = bpp_fin[bpp_fin['codigo_conta'] == '2.08'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
receita      = dre_fin[dre_fin['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
result_bruto = dre_fin[dre_fin['codigo_conta'] == '3.03'][['empresa', 'valor']].rename(columns={'valor': 'result_bruto'})
desp_pessoal = dre_fin[dre_fin['codigo_conta'] == '3.04.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_pessoal'})
desp_admin   = dre_fin[dre_fin['codigo_conta'] == '3.04.03'][['empresa', 'valor']].rename(columns={'valor': 'desp_admin'})
rec_servicos = dre_fin[dre_fin['codigo_conta'] == '3.04.01'][['empresa', 'valor']].rename(columns={'valor': 'rec_servicos'})
outras_rec   = dre_fin[dre_fin['codigo_conta'] == '3.04.05'][['empresa', 'valor']].rename(columns={'valor': 'outras_rec'})

# Juntar tudo
kpis = ll.merge(pl,           on='empresa')
kpis = kpis.merge(receita,      on='empresa')
kpis = kpis.merge(result_bruto, on='empresa')
kpis = kpis.merge(desp_pessoal, on='empresa')
kpis = kpis.merge(desp_admin,   on='empresa')
kpis = kpis.merge(rec_servicos, on='empresa', how='left')
kpis = kpis.merge(outras_rec,   on='empresa', how='left')
kpis[['rec_servicos', 'outras_rec']] = kpis[['rec_servicos', 'outras_rec']].fillna(0)

# Calcular KPIs
kpis['roe']               = kpis['lucro_liquido'] / kpis['patrimonio_liquido']
kpis['margem_liquida']    = kpis['lucro_liquido'] / kpis['receita']
kpis['produto_bancario']  = kpis['result_bruto'] + kpis['rec_servicos'] + kpis['outras_rec']
kpis['indice_eficiencia'] = (kpis['desp_pessoal'].abs() + kpis['desp_admin'].abs()) / kpis['produto_bancario'].abs()

# Adicionar setor, tipo e ano
mapa = {
    'BCO BRASIL S.A.':              ('Financeiro', 'Estatal'),
    'ITAU UNIBANCO HOLDING S.A.':   ('Financeiro', 'Privado'),
    'BCO BRADESCO S.A.':            ('Financeiro', 'Privado'),
    'BCO SANTANDER (BRASIL) S.A.':  ('Financeiro', 'Privado'),
    'BCO BTG PACTUAL S.A.':         ('Financeiro', 'Privado'),
    'BCO ABC BRASIL S.A.':          ('Financeiro', 'Privado'),
    'BCO PAN S.A.':                 ('Financeiro', 'Privado'),
}
kpis['setor'] = kpis['empresa'].map(lambda x: mapa[x][0])
kpis['tipo']  = kpis['empresa'].map(lambda x: mapa[x][1])
kpis['ano']   = 2014

# Exibir
pd.options.display.float_format = '{:.4f}'.format
print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'indice_eficiencia']].sort_values('indice_eficiencia').to_string())

                       empresa     tipo    roe  margem_liquida  indice_eficiencia
4            BCO BRADESCO S.A.  Privado 0.1861          0.1536             0.3916
6  BCO SANTANDER (BRASIL) S.A.  Privado 0.0716          0.0955             0.3988
1          BCO ABC BRASIL S.A.  Privado 0.1437          0.1970             0.4045
5   ITAU UNIBANCO HOLDING S.A.  Privado 0.2142          0.1670             0.4642
0              BCO BRASIL S.A.  Estatal 0.1387          0.0860             0.4938
2         BCO BTG PACTUAL S.A.  Privado 0.1366          0.1821             0.6483
3                 BCO PAN S.A.  Privado 0.0226          0.0120             0.6811


In [8]:

#Cálculo do Z-Score para o BB em relação aos privados
from scipy import stats

# Separar privados e estatais
privados = kpis[kpis['tipo'] == 'Privado']
estatais = kpis[kpis['tipo'] == 'Estatal']

# Calcular média e desvio padrão dos privados
for kpi in ['roe', 'margem_liquida', 'indice_eficiencia']:
    media = privados[kpi].mean()
    desvio = privados[kpi].std()
    
    print(f"\n=== {kpi.upper()} ===")
    print(f"Média privados:  {media:.2f}")
    print(f"Desvio padrão:   {desvio:.2f}")
    
    # Z-Score do BB
    valor_bb = estatais[kpi].values[0]
    zscore = (valor_bb - media) / desvio
    print(f"Valor BB:        {valor_bb:.2f}")
    print(f"Z-Score BB:      {zscore:.2f}")


=== ROE ===
Média privados:  0.13
Desvio padrão:   0.07
Valor BB:        0.14
Z-Score BB:      0.13

=== MARGEM_LIQUIDA ===
Média privados:  0.13
Desvio padrão:   0.07
Valor BB:        0.09
Z-Score BB:      -0.70

=== INDICE_EFICIENCIA ===
Média privados:  0.50
Desvio padrão:   0.13
Valor BB:        0.49
Z-Score BB:      -0.03


In [9]:
import pandas as pd
from scipy import stats

pd.options.display.float_format = '{:.2f}'.format

# ============================================================
# TABELA FINAL — KPIs + Z-Score | Setor Financeiro | 2014
# ============================================================

privados = kpis[kpis['tipo'] == 'Privado']
estatais = kpis[kpis['tipo'] == 'Estatal']

kpis_analisar = ['roe', 'margem_liquida', 'indice_eficiencia']

# Tabela 1 — KPIs de todas as empresas
print("=" * 70)
print("KPIs — SETOR FINANCEIRO 2014")
print("=" * 70)
print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'indice_eficiencia']]
      .sort_values('roe', ascending=False)
      .to_string(index=False))

# Tabela 2 — Médias e Desvio Padrão privadas
print("\n" + "=" * 70)
print("BENCHMARK PRIVADO (média e desvio padrão)")
print("=" * 70)
benchmark_media  = privados[kpis_analisar].mean()
benchmark_desvio = privados[kpis_analisar].std()
print(f"  {'KPI':<25} {'Média':>8} {'Desvio Padrão':>15}")
print(f"  {'-'*50}")
for kpi in kpis_analisar:
    print(f"  {kpi:<25} {benchmark_media[kpi]:>8.2f} {benchmark_desvio[kpi]:>15.2f}")

# Tabela 3 — Z-Score do BB
print("\n" + "=" * 70)
print("Z-SCORE BANCO DO BRASIL vs PRIVADOS — 2014")
print("=" * 70)

bb = estatais.iloc[0]
linhas = []
for kpi in kpis_analisar:
    media  = privados[kpi].mean()
    desvio = privados[kpi].std()
    valor  = bb[kpi]
    zscore = (valor - media) / desvio

    if kpi == 'indice_eficiencia':
        if zscore < -0.5:
            sinal = '✅ Mais eficiente que a média'
        elif zscore > 1:
            sinal = '🚨 Alerta — ineficiente'
        elif zscore > 0.5:
            sinal = '⚠️  Abaixo da média'
        else:
            sinal = '➡️  Na média'
    else:
        if zscore > 0.5:
            sinal = '✅ Acima da média'
        elif zscore < -1:
            sinal = '🚨 Alerta'
        elif zscore < -0.5:
            sinal = '⚠️  Abaixo da média'
        else:
            sinal = '➡️  Na média'

    linhas.append({
        'KPI':            kpi,
        'BB':             round(valor,  2),
        'Média Privados': round(media,  2),
        'Desvio Padrão':  round(desvio, 2),
        'Z-Score':        round(zscore, 2),
        'Sinal':          sinal
    })

resultado = pd.DataFrame(linhas)
print(resultado.to_string(index=False))

KPIs — SETOR FINANCEIRO 2014
                    empresa    tipo  roe  margem_liquida  indice_eficiencia
 ITAU UNIBANCO HOLDING S.A. Privado 0.21            0.17               0.46
          BCO BRADESCO S.A. Privado 0.19            0.15               0.39
        BCO ABC BRASIL S.A. Privado 0.14            0.20               0.40
            BCO BRASIL S.A. Estatal 0.14            0.09               0.49
       BCO BTG PACTUAL S.A. Privado 0.14            0.18               0.65
BCO SANTANDER (BRASIL) S.A. Privado 0.07            0.10               0.40
               BCO PAN S.A. Privado 0.02            0.01               0.68

BENCHMARK PRIVADO (média e desvio padrão)
  KPI                          Média   Desvio Padrão
  --------------------------------------------------
  roe                           0.13            0.07
  margem_liquida                0.13            0.07
  indice_eficiencia             0.50            0.13

Z-SCORE BANCO DO BRASIL vs PRIVADOS — 2014
           

In [10]:
# Buscar contas relacionadas a cada KPI novo
termos = {
    'NIM':            ['intermediação', 'juros', 'spread'],
    'Inadimplência':  ['provisão', 'perdas', 'crédito', 'PDD', 'duvidoso'],
    'Alavancagem':    ['passivo total', 'passivo'],
}

for kpi, palavras in termos.items():
    print(f"\n=== {kpi} ===")
    for palavra in palavras:
        # Buscar na DRE
        dre_match = dre_fin[dre_fin['descricao_conta'].str.contains(palavra, case=False, na=False)][['empresa', 'codigo_conta', 'descricao_conta']].drop_duplicates(subset=['codigo_conta', 'descricao_conta'])
        if len(dre_match) > 0:
            print(f"  [DRE] '{palavra}':")
            print(dre_match.to_string(index=False))

        # Buscar no BPP
        bpp_match = bpp_fin[bpp_fin['descricao_conta'].str.contains(palavra, case=False, na=False)][['empresa', 'codigo_conta', 'descricao_conta']].drop_duplicates(subset=['codigo_conta', 'descricao_conta'])
        if len(bpp_match) > 0:
            print(f"  [BPP] '{palavra}':")
            print(bpp_match.to_string(index=False))

        # Buscar no BPA
        bpa_match = bpa_fin[bpa_fin['descricao_conta'].str.contains(palavra, case=False, na=False)][['empresa', 'codigo_conta', 'descricao_conta']].drop_duplicates(subset=['codigo_conta', 'descricao_conta'])
        if len(bpa_match) > 0:
            print(f"  [BPA] '{palavra}':")
            print(bpa_match.to_string(index=False))


=== NIM ===
  [DRE] 'intermediação':
          empresa codigo_conta                          descricao_conta
  BCO BRASIL S.A.         3.01     Receitas da Intermediação Financeira
  BCO BRASIL S.A.         3.02     Despesas da Intermediação Financeira
  BCO BRASIL S.A.         3.03 Resultado Bruto Intermediação Financeira
BCO BRADESCO S.A.      3.02.03      Despesa de Intermediação Financeira
  [DRE] 'juros':
                    empresa codigo_conta                 descricao_conta
            BCO BRASIL S.A.      3.01.01                Receita de Juros
            BCO BRASIL S.A.      3.02.01               Despesas de Juros
       BCO BTG PACTUAL S.A.      3.01.01              Receitas com Juros
       BCO BTG PACTUAL S.A.      3.02.01              Despesas com Juros
               BCO PAN S.A.      3.01.01  Receitas com juros e similares
               BCO PAN S.A.      3.02.01  Despesas com juros e similares
          BCO BRADESCO S.A.      3.01.01    Receita de Juros e Similares
 

In [11]:
# ============================================================
# NIM — Net Interest Margin
# ============================================================

# Extrair Receita de Juros (3.01.01) e Despesa de Juros (3.02.01)
rec_juros  = dre_fin[dre_fin['codigo_conta'] == '3.01.01'][['empresa', 'valor']].rename(columns={'valor': 'rec_juros'})
desp_juros = dre_fin[dre_fin['codigo_conta'] == '3.02.01'][['empresa', 'valor']].rename(columns={'valor': 'desp_juros'})

# Extrair Ativo Total do BPA (conta 1)
ativo_total = bpa_fin[bpa_fin['codigo_conta'] == '1'][['empresa', 'valor']].rename(columns={'valor': 'ativo_total'})

print("=== Receita de Juros ===")
print(rec_juros.to_string())
print("\n=== Despesa de Juros ===")
print(desp_juros.to_string())
print("\n=== Ativo Total ===")
print(ativo_total.to_string())

=== Receita de Juros ===
                         empresa    rec_juros
1                BCO BRASIL S.A. 137778601.00
437         BCO BTG PACTUAL S.A.   6721169.00
514                 BCO PAN S.A.   6889029.00
554            BCO BRADESCO S.A. 103893096.00
593   ITAU UNIBANCO HOLDING S.A. 120115000.00
666  BCO SANTANDER (BRASIL) S.A.  58923916.00

=== Despesa de Juros ===
                         empresa   desp_juros
3                BCO BRASIL S.A. -91124202.00
441         BCO BTG PACTUAL S.A.  -9417973.00
516                 BCO PAN S.A.  -4104049.00
559            BCO BRADESCO S.A.         0.00
597   ITAU UNIBANCO HOLDING S.A. -72977000.00
668  BCO SANTANDER (BRASIL) S.A. -31695404.00

=== Ativo Total ===
                          empresa   ativo_total
0                 BCO BRASIL S.A. 1278136948.00
704           BCO ABC BRASIL S.A.   19652604.00
734          BCO BTG PACTUAL S.A.  157712093.00
828                  BCO PAN S.A.   25579362.00
865             BCO BRADESCO S.A.  930451016

In [12]:
# Usando contas agregadas — existem em todas as 7 empresas
rec_intermediacao  = dre_fin[dre_fin['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'rec_intermediacao'})
desp_intermediacao = dre_fin[dre_fin['codigo_conta'] == '3.02'][['empresa', 'valor']].rename(columns={'valor': 'desp_intermediacao'})
ativo_total        = bpa_fin[bpa_fin['codigo_conta'] == '1'][['empresa', 'valor']].rename(columns={'valor': 'ativo_total'})

# Juntar tudo
nim_df = rec_intermediacao.merge(desp_intermediacao, on='empresa')
nim_df = nim_df.merge(ativo_total, on='empresa')

# Calcular NIM
# Despesa é negativa, por isso subtraímos
nim_df['nim'] = (nim_df['rec_intermediacao'] + nim_df['desp_intermediacao']) / nim_df['ativo_total']

print(nim_df[['empresa', 'rec_intermediacao', 'desp_intermediacao', 'ativo_total', 'nim']]
      .sort_values('nim', ascending=False)
      .to_string(index=False))


                    empresa  rec_intermediacao  desp_intermediacao   ativo_total  nim
               BCO PAN S.A.         6889029.00         -4736168.00   25579362.00 0.08
BCO SANTANDER (BRASIL) S.A.        58923916.00        -31695404.00  520230910.00 0.05
 ITAU UNIBANCO HOLDING S.A.       129035000.00        -72977000.00 1127203000.00 0.05
          BCO BRADESCO S.A.        99723519.00        -53847329.00  930451016.00 0.05
            BCO BRASIL S.A.       137778601.00       -105909418.00 1278136948.00 0.02
       BCO BTG PACTUAL S.A.        11626540.00         -9885826.00  157712093.00 0.01
        BCO ABC BRASIL S.A.         1666916.00         -1477428.00   19652604.00 0.01


In [13]:
kpis = kpis.merge(nim_df[['empresa', 'nim']], on='empresa')

print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'indice_eficiencia', 'nim']]
      .sort_values('roe', ascending=False)
      .to_string(index=False))


                    empresa    tipo  roe  margem_liquida  indice_eficiencia  nim
 ITAU UNIBANCO HOLDING S.A. Privado 0.21            0.17               0.46 0.05
          BCO BRADESCO S.A. Privado 0.19            0.15               0.39 0.05
        BCO ABC BRASIL S.A. Privado 0.14            0.20               0.40 0.01
            BCO BRASIL S.A. Estatal 0.14            0.09               0.49 0.02
       BCO BTG PACTUAL S.A. Privado 0.14            0.18               0.65 0.01
BCO SANTANDER (BRASIL) S.A. Privado 0.07            0.10               0.40 0.05
               BCO PAN S.A. Privado 0.02            0.01               0.68 0.08


In [14]:
# Passivo Total = Ativo Total - Patrimônio Líquido
# (pois nem todos os bancos têm conta 2 explícita no BPP)

pl          = bpp_fin[bpp_fin['codigo_conta'] == '2.08'][['empresa', 'valor']].rename(columns={'valor': 'patrimonio_liquido'})
ativo_total = bpa_fin[bpa_fin['codigo_conta'] == '1'][['empresa', 'valor']].rename(columns={'valor': 'ativo_total'})

alavancagem_df = ativo_total.merge(pl, on='empresa')

# Passivo Total = Ativo Total - PL
alavancagem_df['passivo_total']  = alavancagem_df['ativo_total'] - alavancagem_df['patrimonio_liquido']
alavancagem_df['alavancagem']    = alavancagem_df['passivo_total'] / alavancagem_df['patrimonio_liquido']

print(alavancagem_df[['empresa', 'ativo_total', 'patrimonio_liquido', 'passivo_total', 'alavancagem']]
      .sort_values('alavancagem', ascending=False)
      .to_string(index=False))

                    empresa   ativo_total  patrimonio_liquido  passivo_total  alavancagem
            BCO BRASIL S.A. 1278136948.00         85440036.00  1192696912.00        13.96
          BCO BRADESCO S.A.  930451016.00         82291805.00   848159211.00        10.31
 ITAU UNIBANCO HOLDING S.A. 1127203000.00        100617000.00  1026586000.00        10.20
       BCO BTG PACTUAL S.A.  157712093.00         15495119.00   142216974.00         9.18
        BCO ABC BRASIL S.A.   19652604.00          2284769.00    17367835.00         7.60
               BCO PAN S.A.   25579362.00          3643526.00    21935836.00         6.02
BCO SANTANDER (BRASIL) S.A.  520230910.00         78683293.00   441547617.00         5.61


In [15]:
kpis = kpis.merge(alavancagem_df[['empresa', 'alavancagem']], on='empresa')

print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'indice_eficiencia', 'nim', 'alavancagem']]
      .sort_values('roe', ascending=False)
      .to_string(index=False))

                    empresa    tipo  roe  margem_liquida  indice_eficiencia  nim  alavancagem
 ITAU UNIBANCO HOLDING S.A. Privado 0.21            0.17               0.46 0.05        10.20
          BCO BRADESCO S.A. Privado 0.19            0.15               0.39 0.05        10.31
        BCO ABC BRASIL S.A. Privado 0.14            0.20               0.40 0.01         7.60
            BCO BRASIL S.A. Estatal 0.14            0.09               0.49 0.02        13.96
       BCO BTG PACTUAL S.A. Privado 0.14            0.18               0.65 0.01         9.18
BCO SANTANDER (BRASIL) S.A. Privado 0.07            0.10               0.40 0.05         5.61
               BCO PAN S.A. Privado 0.02            0.01               0.68 0.08         6.02


In [16]:
# Mapeamento manual das contas de PDD por empresa
contas_pdd = {
    'BCO BRASIL S.A.':              ['3.02.02', '3.02.03'],
    'BCO BRADESCO S.A.':            ['3.04.06.01'],
    'ITAU UNIBANCO HOLDING S.A.':   ['3.04.06.01'],
    'BCO SANTANDER (BRASIL) S.A.':  ['3.04.06.01', '3.04.06.03'],
    'BCO BTG PACTUAL S.A.':         ['3.02.02'],
    'BCO ABC BRASIL S.A.':          ['3.04.06.01'],
    'BCO PAN S.A.':                 ['3.02.02'],
}

# Calcular PDD total por empresa
linhas = []
for empresa, contas in contas_pdd.items():
    df_emp = dre_fin[
        (dre_fin['empresa'] == empresa) &
        (dre_fin['codigo_conta'].isin(contas))
    ]
    pdd_total = df_emp['valor'].sum()
    linhas.append({'empresa': empresa, 'pdd': pdd_total})

pdd_df = pd.DataFrame(linhas)

# PDD como % da Receita Total (3.01)
receita = dre_fin[dre_fin['codigo_conta'] == '3.01'][['empresa', 'valor']].rename(columns={'valor': 'receita'})
pdd_df = pdd_df.merge(receita, on='empresa')
pdd_df['pdd_ratio'] = pdd_df['pdd'].abs() / pdd_df['receita'].abs()

print(pdd_df[['empresa', 'pdd', 'receita', 'pdd_ratio']]
      .sort_values('pdd_ratio', ascending=False)
      .to_string(index=False))

                    empresa          pdd      receita  pdd_ratio
BCO SANTANDER (BRASIL) S.A. -12633734.00  58923916.00       0.21
 ITAU UNIBANCO HOLDING S.A. -18832000.00 129035000.00       0.15
               BCO PAN S.A.   -908542.00   6889029.00       0.13
            BCO BRASIL S.A. -14785216.00 137778601.00       0.11
          BCO BRADESCO S.A. -10291386.00  99723519.00       0.10
        BCO ABC BRASIL S.A.    -74705.00   1666916.00       0.04
       BCO BTG PACTUAL S.A.   -467853.00  11626540.00       0.04


In [19]:
kpis = kpis.merge(pdd_df[['empresa', 'pdd_ratio']], on='empresa')

print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'indice_eficiencia', 'nim', 'alavancagem', 'pdd_ratio']]
      .sort_values('roe', ascending=False)
      .to_string(index=False))

                    empresa    tipo  roe  margem_liquida  indice_eficiencia  nim  alavancagem  pdd_ratio
 ITAU UNIBANCO HOLDING S.A. Privado 0.21            0.17               0.46 0.05        10.20       0.15
          BCO BRADESCO S.A. Privado 0.19            0.15               0.39 0.05        10.31       0.10
        BCO ABC BRASIL S.A. Privado 0.14            0.20               0.40 0.01         7.60       0.04
            BCO BRASIL S.A. Estatal 0.14            0.09               0.49 0.02        13.96       0.11
       BCO BTG PACTUAL S.A. Privado 0.14            0.18               0.65 0.01         9.18       0.04
BCO SANTANDER (BRASIL) S.A. Privado 0.07            0.10               0.40 0.05         5.61       0.21
               BCO PAN S.A. Privado 0.02            0.01               0.68 0.08         6.02       0.13


In [20]:
# ============================================================
# TABELA FINAL — 6 KPIs + Z-Score | Setor Financeiro | 2014
# ============================================================

privados = kpis[kpis['tipo'] == 'Privado']
estatais = kpis[kpis['tipo'] == 'Estatal']
bb = estatais.iloc[0]

# KPIs e se maior = melhor (True) ou menor = melhor (False)
kpis_config = {
    'roe':               True,
    'margem_liquida':    True,
    'nim':               True,
    'indice_eficiencia': False,
    'alavancagem':       False,
    'pdd_ratio':         False,
}

print("=" * 75)
print("KPIs — SETOR FINANCEIRO 2014")
print("=" * 75)
print(kpis[['empresa', 'tipo', 'roe', 'margem_liquida', 'nim', 'indice_eficiencia', 'alavancagem', 'pdd_ratio']]
      .sort_values('roe', ascending=False)
      .to_string(index=False))

print("\n" + "=" * 75)
print("Z-SCORE BANCO DO BRASIL vs PRIVADOS — 2014")
print("=" * 75)

linhas = []
for kpi, maior_melhor in kpis_config.items():
    media  = privados[kpi].mean()
    desvio = privados[kpi].std()
    valor  = bb[kpi]
    zscore = (valor - media) / desvio

    # Inverter sinal para KPIs onde menor = melhor
    zscore_interpretado = zscore if maior_melhor else -zscore

    if zscore_interpretado > 0.5:
        sinal = '✅ Acima da média'
    elif zscore_interpretado < -1:
        sinal = '🚨 Alerta'
    elif zscore_interpretado < -0.5:
        sinal = '⚠️  Abaixo da média'
    else:
        sinal = '➡️  Na média'

    linhas.append({
        'KPI':            kpi,
        'BB':             round(valor,  2),
        'Média Privados': round(media,  2),
        'Desvio Padrão':  round(desvio, 2),
        'Z-Score':        round(zscore, 2),
        'Sinal':          sinal
    })

resultado = pd.DataFrame(linhas)
print(resultado.to_string(index=False))

# Score geral — média dos Z-Scores interpretados
zscores_interpretados = []
for kpi, maior_melhor in kpis_config.items():
    media  = privados[kpi].mean()
    desvio = privados[kpi].std()
    zscore = (bb[kpi] - media) / desvio
    zscores_interpretados.append(zscore if maior_melhor else -zscore)

score_geral = sum(zscores_interpretados) / len(zscores_interpretados)
print(f"\n{'=' * 75}")
print(f"SCORE GERAL BB — 2014: {score_geral:.2f}")
if score_geral > 0.5:
    print("✅ BB performa ACIMA da média privada")
elif score_geral < -0.5:
    print("🚨 BB performa ABAIXO da média privada")
else:
    print("➡️  BB performa NA MÉDIA dos privados")
print("=" * 75)

KPIs — SETOR FINANCEIRO 2014
                    empresa    tipo  roe  margem_liquida  nim  indice_eficiencia  alavancagem  pdd_ratio
 ITAU UNIBANCO HOLDING S.A. Privado 0.21            0.17 0.05               0.46        10.20       0.15
          BCO BRADESCO S.A. Privado 0.19            0.15 0.05               0.39        10.31       0.10
        BCO ABC BRASIL S.A. Privado 0.14            0.20 0.01               0.40         7.60       0.04
            BCO BRASIL S.A. Estatal 0.14            0.09 0.02               0.49        13.96       0.11
       BCO BTG PACTUAL S.A. Privado 0.14            0.18 0.01               0.65         9.18       0.04
BCO SANTANDER (BRASIL) S.A. Privado 0.07            0.10 0.05               0.40         5.61       0.21
               BCO PAN S.A. Privado 0.02            0.01 0.08               0.68         6.02       0.13

Z-SCORE BANCO DO BRASIL vs PRIVADOS — 2014
              KPI    BB  Média Privados  Desvio Padrão  Z-Score               Sinal
   

In [22]:
import pandas as pd

kpis = pd.read_csv("../KPIs/kpis_financeiro.csv", sep=";", encoding="utf-8")

# Ver quais empresas aparecem em cada ano
pivot = kpis[kpis['tipo'] == 'Privado'].groupby(['empresa', 'ano']).size().unstack(fill_value=0)
print(pivot.to_string())


ano                          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023
empresa                                                                                
BCO ABC BRASIL S.A.             1     1     1     1     1     1     0     0     0     0
BCO BRADESCO S.A.               1     1     1     1     1     1     0     0     0     0
BCO BTG PACTUAL S.A.            1     1     1     1     1     1     1     1     1     1
BCO PAN S.A.                    1     1     1     1     1     1     1     1     1     1
BCO SANTANDER (BRASIL) S.A.     1     1     1     1     1     1     0     0     0     0
ITAU UNIBANCO HOLDING S.A.      1     1     1     1     1     1     1     1     1     1


In [24]:
dre_2020 = pd.read_csv("../DRE_tratado/dfp_dre_2020_filtrado.csv", sep=";", encoding="utf-8")

empresas_problema = ['BCO ABC BRASIL S.A.', 'BCO BRADESCO S.A.', 'BCO SANTANDER (BRASIL) S.A.']

for empresa in empresas_problema:
    print(f"\n=== {empresa} — contas 3.09 e 3.11 ===")
    contas = dre_2020[
        (dre_2020['empresa'] == empresa) &
        (dre_2020['codigo_conta'].str.match(r'^3\.(09|10|11)'))
    ][['codigo_conta', 'descricao_conta', 'valor']]
    print(contas.sort_values('codigo_conta').to_string(index=False))


=== BCO ABC BRASIL S.A. — contas 3.09 e 3.11 ===
codigo_conta                                                        descricao_conta     valor
        3.09 Lucro ou Prejuízo antes das Participações e Contribuições Estatutárias 340688.00
     3.09.01                             Atribuído a Sócios da Empresa Controladora 340688.00
     3.09.02                                   Atribuído a Sócios Não Controladores      0.00
        3.10                  Participações nos Lucros e Contribuições Estatutárias      0.00
        3.11                       Lucro ou Prejuízo Líquido Consolidado do Período 340688.00
     3.11.01                           Atribuído aos Sócios da Empresa Controladora 340688.00
     3.11.02                                 Atribuído aos Sócios não Controladores      0.00

=== BCO BRADESCO S.A. — contas 3.09 e 3.11 ===
codigo_conta                                                        descricao_conta       valor
        3.09 Lucro ou Prejuízo antes das Participações

In [26]:
import pandas as pd

dre_2020 = pd.read_csv("../DRE_tratado/dfp_dre_2020_filtrado.csv", sep=";", encoding="utf-8")
bpa_2020 = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2020_filtrado.csv", sep=";", encoding="utf-8")
bpp_2020 = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2020_filtrado.csv", sep=";", encoding="utf-8")

fin_dre = dre_2020[dre_2020['setor'] == 'Financeiro']
fin_bpa = bpa_2020[bpa_2020['setor'] == 'Financeiro']
fin_bpp = bpp_2020[bpp_2020['setor'] == 'Financeiro']

print("=== EMPRESAS NA DRE 2020 ===")
print(fin_dre['empresa'].unique())

print("\n=== EMPRESAS NO BPA 2020 ===")
print(fin_bpa['empresa'].unique())

print("\n=== EMPRESAS NO BPP 2020 ===")
print(fin_bpp['empresa'].unique())

=== EMPRESAS NA DRE 2020 ===
<StringArray>
[            'BCO BRASIL S.A.',         'BCO ABC BRASIL S.A.',
        'BCO BTG PACTUAL S.A.',                'BCO PAN S.A.',
           'BCO BRADESCO S.A.',  'ITAU UNIBANCO HOLDING S.A.',
 'BCO SANTANDER (BRASIL) S.A.']
Length: 7, dtype: str

=== EMPRESAS NO BPA 2020 ===
<StringArray>
[            'BCO BRASIL S.A.',         'BCO ABC BRASIL S.A.',
        'BCO BTG PACTUAL S.A.',                'BCO PAN S.A.',
           'BCO BRADESCO S.A.',  'ITAU UNIBANCO HOLDING S.A.',
 'BCO SANTANDER (BRASIL) S.A.']
Length: 7, dtype: str

=== EMPRESAS NO BPP 2020 ===
<StringArray>
[            'BCO BRASIL S.A.',         'BCO ABC BRASIL S.A.',
        'BCO BTG PACTUAL S.A.',                'BCO PAN S.A.',
           'BCO BRADESCO S.A.',  'ITAU UNIBANCO HOLDING S.A.',
 'BCO SANTANDER (BRASIL) S.A.']
Length: 7, dtype: str


In [27]:
dre_fin = fin_dre
bpa_fin = fin_bpa
bpp_fin = fin_bpp

extrações = {
    'll_3.09.01':    dre_fin[dre_fin['codigo_conta'] == '3.09.01']['empresa'].tolist(),
    'll_3.11.01':    dre_fin[dre_fin['codigo_conta'] == '3.11.01']['empresa'].tolist(),
    'pl_2.08':       bpp_fin[bpp_fin['codigo_conta'] == '2.08']['empresa'].tolist(),
    'receita_3.01':  dre_fin[dre_fin['codigo_conta'] == '3.01']['empresa'].tolist(),
    'res_bruto_3.03':dre_fin[dre_fin['codigo_conta'] == '3.03']['empresa'].tolist(),
    'desp_pes_3.04.02':dre_fin[dre_fin['codigo_conta'] == '3.04.02']['empresa'].tolist(),
    'desp_adm_3.04.03':dre_fin[dre_fin['codigo_conta'] == '3.04.03']['empresa'].tolist(),
    'ativo_total_1': bpa_fin[bpa_fin['codigo_conta'] == '1']['empresa'].tolist(),
}

todas = set(dre_fin['empresa'].unique())

for nome, empresas in extrações.items():
    faltando = todas - set(empresas)
    if faltando:
        print(f"❌ {nome}: faltando {faltando}")
    else:
        print(f"✅ {nome}: todas as 7 empresas")

❌ ll_3.09.01: faltando {'BCO BRADESCO S.A.', 'BCO BRASIL S.A.'}
❌ ll_3.11.01: faltando {'BCO BTG PACTUAL S.A.', 'ITAU UNIBANCO HOLDING S.A.', 'BCO PAN S.A.'}
❌ pl_2.08: faltando {'BCO ABC BRASIL S.A.', 'BCO SANTANDER (BRASIL) S.A.', 'BCO BRADESCO S.A.', 'BCO BRASIL S.A.'}
✅ receita_3.01: todas as 7 empresas
✅ res_bruto_3.03: todas as 7 empresas
✅ desp_pes_3.04.02: todas as 7 empresas
✅ desp_adm_3.04.03: todas as 7 empresas
✅ ativo_total_1: todas as 7 empresas


In [28]:
empresas_sem_pl = ['BCO ABC BRASIL S.A.', 'BCO SANTANDER (BRASIL) S.A.', 
                   'BCO BRADESCO S.A.', 'BCO BRASIL S.A.']

for empresa in empresas_sem_pl:
    print(f"\n=== {empresa} — contas 2.08.xx ===")
    contas = bpp_fin[
        (bpp_fin['empresa'] == empresa) &
        (bpp_fin['codigo_conta'].str.startswith('2.'))
    ][['codigo_conta', 'descricao_conta', 'valor']]
    print(contas.sort_values('codigo_conta').to_string(index=False))


=== BCO ABC BRASIL S.A. — contas 2.08.xx ===
 codigo_conta                                                    descricao_conta       valor
         2.01 Passivos Financeiros Avaliados ao Valor Justo através do Resultado  1973096.00
      2.01.01                                                 Dívida subordinada        0.00
      2.01.02                                   Captações - Repasses no exterior        0.00
      2.01.03                                                        Derivativos  1973096.00
         2.02                           Passivos Financeiros ao Custo Amortizado 32884049.00
      2.02.01                                                          Depósitos        0.00
      2.02.02                                        Captações no Mercado Aberto        0.00
      2.02.03                                   Recursos Mercado Interfinanceiro        0.00
      2.02.04                                                   Outras Captações 32884049.00
   2.02.04.01           

In [29]:
import pandas as pd

kpis = pd.read_csv("../KPIs/kpis_financeiro.csv", sep=";", encoding="utf-8")

pivot = kpis[kpis['tipo'] == 'Privado'].groupby(['empresa', 'ano']).size().unstack(fill_value=0)
print(pivot.to_string())

ano                          2014  2015  2016  2017  2018  2019  2020  2021  2022  2023
empresa                                                                                
BCO ABC BRASIL S.A.             1     1     1     1     1     1     1     1     1     1
BCO BRADESCO S.A.               1     1     1     1     1     1     1     1     1     1
BCO BTG PACTUAL S.A.            1     1     1     1     1     1     1     1     1     1
BCO PAN S.A.                    1     1     1     1     1     1     1     1     1     1
BCO SANTANDER (BRASIL) S.A.     1     1     1     1     1     1     1     0     1     1
ITAU UNIBANCO HOLDING S.A.      1     1     1     1     1     1     1     1     1     1


In [30]:
dre_2021 = pd.read_csv("../DRE_tratado/dfp_dre_2021_filtrado.csv", sep=";", encoding="utf-8")
bpa_2021 = pd.read_csv("../Balanco_ativo_tratado/dfp_bp_2021_filtrado.csv", sep=";", encoding="utf-8")
bpp_2021 = pd.read_csv("../Balanco_passivo_tratado/dfp_bpp_2021_filtrado.csv", sep=";", encoding="utf-8")

sant_dre = dre_2021[dre_2021['empresa'] == 'BCO SANTANDER (BRASIL) S.A.']
sant_bpp = bpp_2021[bpp_2021['empresa'] == 'BCO SANTANDER (BRASIL) S.A.']

print("=== SANTANDER DRE 2021 — contas relevantes ===")
contas_checar = ['3.01', '3.02', '3.03', '3.04.02', '3.04.03', '3.09.01', '3.11.01']
print(sant_dre[sant_dre['codigo_conta'].isin(contas_checar)][['codigo_conta', 'descricao_conta', 'valor']].to_string())

print("\n=== SANTANDER BPP 2021 — Patrimônio ===")
print(sant_bpp[sant_bpp['codigo_conta'].isin(['2.07', '2.08'])][['codigo_conta', 'descricao_conta', 'valor']].to_string())

=== SANTANDER DRE 2021 — contas relevantes ===
    codigo_conta                               descricao_conta        valor
653         3.01          Receitas de Intermediação Financeira  77987308.00
654         3.02          Despesas de Intermediação Financeira -26668842.00
655         3.03   Resultado Bruto de Intermediação Financeira  51318466.00
658      3.04.03                          Despesas com Pessoal  -9025702.00
687      3.11.01  Atribuído aos Sócios da Empresa Controladora  15528052.00

=== SANTANDER BPP 2021 — Patrimônio ===
     codigo_conta                 descricao_conta        valor
1545         2.07  Patrimônio Líquido Consolidado 105974495.00


In [31]:
print(sant_dre[sant_dre['codigo_conta'].str.startswith('3.04')][['codigo_conta', 'descricao_conta', 'valor']].sort_values('codigo_conta').to_string())

    codigo_conta                                                                                       descricao_conta        valor
656         3.04                                                               Outras Despesas e Receitas Operacionais -26568138.00
657      3.04.01                                         Despesa de Provisão para Perda Esperada para Risco de Crédito         0.00
658      3.04.03                                                                                  Despesas com Pessoal  -9025702.00
659      3.04.04                                                                    Outras Despesas de Administrativas -30150076.00
660   3.04.04.01                                                                       Outras despesas administrativas  -8290717.00
661   3.04.04.02                                                                                        Ativo tangível  -1850780.00
662   3.04.04.03                                                            

In [1]:
import pandas as pd

# Verificar presença de bancos estatais em todos os anos
bancos_estatais = [
    'BRB',
    'BANRISUL',
    'BANESTES',
    'NORDESTE',
    'AMAZONIA',
    'BASA',
]

print("=== COBERTURA DOS BANCOS ESTATAIS POR ANO ===\n")

for ano in range(2014, 2024):
    # Ler arquivo bruto da DRE
    dre = pd.read_csv(
        f"../DRE_bruto/dfp_dre_{ano}.csv",
        sep=";",
        encoding="latin1"
    )
    
    print(f"--- {ano} ---")
    for termo in bancos_estatais:
        found = dre[
            dre['DENOM_CIA'].str.contains(termo, case=False, na=False)
        ]['DENOM_CIA'].unique()
        if len(found) > 0:
            # Decodificar nome corretamente
            nomes = []
            for nome in found:
                try:
                    nomes.append(nome.encode('latin1').decode('utf-8'))
                except:
                    nomes.append(nome)
            print(f"  {termo}: {nomes}")

=== COBERTURA DOS BANCOS ESTATAIS POR ANO ===

--- 2014 ---
  BRB: ['BRB BANCO DE BRASILIA S.A.']
  BANESTES: ['BANESTES S.A. - BCO EST ESPIRITO SANTO']
  NORDESTE: ['PARTICIPAÇÕES INDUST. DO NORDESTE S.A.']
  BASA: ['CIA FERRO LIGAS DA BAHIA - FERBASA']
--- 2015 ---
  BRB: ['BRB BANCO DE BRASILIA S.A.']
  BANESTES: ['BANESTES S.A. - BCO EST ESPIRITO SANTO']
  NORDESTE: ['PARTICIPAÇÕES INDUST. DO NORDESTE S.A.']
  BASA: ['CIA FERRO LIGAS DA BAHIA - FERBASA']
--- 2016 ---
  BRB: ['BRB BANCO DE BRASILIA S.A.']
  BANESTES: ['BANESTES S.A. - BCO EST ESPIRITO SANTO']
  NORDESTE: ['PARTICIPAÇÕES INDUST. DO NORDESTE S.A.']
  BASA: ['CIA FERRO LIGAS DA BAHIA - FERBASA']
--- 2017 ---
  BRB: ['BRB BANCO DE BRASILIA S.A.']
  BANESTES: ['BANESTES S.A. - BCO EST ESPIRITO SANTO']
  NORDESTE: ['DASS NORDESTE CALÇADOS E ARTIGOS ESPORTIVOS S.A', 'PARTICIPAÇÕES INDUST. DO NORDESTE S.A.']
  BASA: ['CIA FERRO LIGAS DA BAHIA - FERBASA']
--- 2018 ---
  BRB: ['BRB BANCO DE BRASILIA S.A.']
  BANESTES: ['BANES